# One-day holding period: exploration (Phase 13)

Exploration gate of the one-day track (see ROADMAP.md). BTC/USDT only. Daily data for the one-day moves; the 15-minute data only for step 3. Existing data; no download.

**Volatility and counts only.** No mean, median or share of positive returns is computed, and no relationship between any condition and price direction.

**Steps**
1. Typical one-day move and cost ratio, overall and by year.
2. The same by day of week of the entry day, for 2017–2023 and 2024 onward.
3. The same grouped by how active the signal day was around the US open (09:00–10:59 New York time), in development-period terciles.

**Definitions (fixed before any result was computed)**
- **One-day trade:** a signal at the close of day t enters at the open of day t + 1 (the entry day) and exits at the open of day t + 2. Its one-day move is log(open of t + 2 / open of t + 1), from daily candles; the daily data has no missing days.
- **Typical move and cost ratio:** typical move = sample standard deviation of these log moves (shown in %); cost ratio = 0.30% / typical move.
- **Year, day of week and period** are those of the entry day (UTC). 2017–2023 = entry day before 2024-01-01; 2024 onward = entry day on or after 2024-01-01.
- **Signal-day activity at the US open:** the realized volatility of 15-minute log returns (each candle's close vs the previous candle's close, as in `add_intraday_features`), summed in squares over the 8 candles opening from 09:00 to 10:45 New York time on the signal day t, then square-rooted. That window ends at 11:00 New York (15:00 or 16:00 UTC), before the close of day t, so it is known at entry. If any of the 8 returns is missing (a missing candle, including the 08:45 candle the first return needs), the value is NaN and the trade is left out of step 3 and counted.
- **Terciles:** the 1/3 and 2/3 quantiles of signal-day activity over development trades only (entry day before 2024-01-01). Low = at or below the first, middle = above the first and at or below the second, high = above the second. The same thresholds are applied to 2024 onward.

## Step 0: Imports and settings

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from cryptosense.data import load_processed
from cryptosense.intraday import NY_TZ, lag_by_time

DATA_DIR = Path.cwd().parent / "data"
SYMBOL = "BTC/USDT"
COST_PER_SIDE = 0.0015
ROUND_TRIP = 2 * COST_PER_SIDE
SPLIT = pd.Timestamp("2024-01-01", tz="UTC")
WINDOW_HOURS_NY = [9, 10]
CANDLES_IN_WINDOW = 8

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 100)

## Step 1: Load data and build one-day moves

In [2]:
daily = load_processed(SYMBOL, DATA_DIR)
m15 = load_processed(SYMBOL, DATA_DIR, timeframe="15m")
print(f"daily: {len(daily):,} days, {daily.index[0].date()} -> {daily.index[-1].date()}")
print(f"daily steps other than 1 day: {int((daily.index.to_series().diff().dropna() != pd.Timedelta(days=1)).sum())}")
print(f"15m:   {len(m15):,} candles, {m15.index[0]} -> {m15.index[-1]}")

opens = daily["open"]
trades = pd.DataFrame({
    "signal_day": daily.index[:-2],
    "entry_day": daily.index[1:-1],
    "move": np.log(opens.to_numpy()[2:] / opens.to_numpy()[1:-1]),
}).set_index("entry_day")
trades["year"] = trades.index.year
trades["weekday"] = trades.index.day_name()
trades["period"] = np.where(trades.index < SPLIT, "2017–2023", "2024 onward")
print(f"one-day trades: {len(trades):,}, entry days {trades.index[0].date()} -> {trades.index[-1].date()}")

daily: 3,352 days, 2017-08-17 -> 2026-10-20
daily steps other than 1 day: 0
15m:   321,296 candles, 2017-08-17 04:00:00+00:00 -> 2026-10-21 21:00:00+00:00
one-day trades: 3,350, entry days 2017-08-18 -> 2026-10-19


In [3]:
def cost_stats(r):
    """n, typical move (std, %) and round-trip cost / typical move."""
    move = r.std()
    return pd.Series({"n": len(r), "typical_move_%": 100 * move, "cost_ratio": ROUND_TRIP / move})


def grouped(frame, by):
    """cost_stats for each group of ``frame`` by ``by``, with n as int."""
    table = frame.groupby(by, sort=False)["move"].apply(cost_stats).unstack()
    table["n"] = table["n"].astype(int)
    return table


def show(table):
    print(table.to_string(formatters={
        col: (lambda v: f"{v:,.0f}") if (col == "n" or (isinstance(col, tuple) and col[-1] == "n"))
        else (lambda v: f"{v:.3f}")
        for col in table.columns
    }))

## Step 2: Typical one-day move and cost ratio, overall and by year

2017 starts with entry day 2017-08-18; the last year is partial.

In [4]:
by_year = grouped(trades, "year")
by_year.index = by_year.index.astype(str)
by_year.loc["overall"] = cost_stats(trades["move"])
by_year["n"] = by_year["n"].astype(int)
show(by_year)

            n typical_move_% cost_ratio
year                                   
2017      136          5.723      0.052
2018      365          4.398      0.068
2019      365          3.568      0.084
2020      366          4.237      0.071
2021      365          4.238      0.071
2022      365          3.371      0.089
2023      365          2.280      0.132
2024      366          2.757      0.109
2025      365          2.181      0.138
2026      292          2.356      0.127
overall 3,350          3.524      0.085


## Step 3: By day of week of the entry day, 2017–2023 and 2024 onward

In [5]:
WEEKDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
by_weekday = pd.concat({
    period: grouped(frame, "weekday").reindex(WEEKDAYS)
    for period, frame in trades.groupby("period")
}, axis=1)
by_weekday.index.name = "entry weekday"
show(by_weekday)

              2017–2023                           2024 onward                          
                      n typical_move_% cost_ratio           n typical_move_% cost_ratio
entry weekday                                                                          
Monday              332          4.339      0.069         147          2.837      0.106
Tuesday             332          3.803      0.079         146          2.529      0.119
Wednesday           332          3.943      0.076         146          2.719      0.110
Thursday            332          5.092      0.059         146          2.638      0.114
Friday              333          3.902      0.077         146          2.704      0.111
Saturday            333          2.719      0.110         146          1.232      0.244
Sunday              333          3.055      0.098         146          2.034      0.147


## Step 4: By signal-day activity around the US open (09:00–10:59 New York)

First the signal-day realized volatility in the window, then development-period terciles, then the typical one-day move per tercile.

In [6]:
log_close = np.log(m15["close"])
ret_15m = log_close - lag_by_time(log_close, "15min")
ny = m15.index.tz_convert(NY_TZ)
in_window = np.isin(ny.hour, WINDOW_HOURS_NY)

window = pd.DataFrame({"sq": ret_15m[in_window] ** 2, "ok": ret_15m[in_window].notna()})
window["day"] = m15.index[in_window].normalize()   # the UTC day; the window lies inside it
per_day = window.groupby("day").agg(sq_sum=("sq", "sum"), returns=("ok", "sum"))
per_day["rv_open"] = np.sqrt(per_day["sq_sum"]).where(per_day["returns"] == CANDLES_IN_WINDOW)
per_day.index = per_day.index.as_unit(trades["signal_day"].dt.unit)

trades["rv_open"] = per_day["rv_open"].reindex(trades["signal_day"]).to_numpy()
missing = trades["rv_open"].isna()
print(f"trades without a complete signal-day window: {int(missing.sum())} "
      f"(2017–2023: {int((missing & (trades['period'] == '2017–2023')).sum())}, "
      f"2024 onward: {int((missing & (trades['period'] == '2024 onward')).sum())})")

dev_rv = trades.loc[(trades["period"] == "2017–2023") & ~missing, "rv_open"]
q1, q2 = dev_rv.quantile(1 / 3), dev_rv.quantile(2 / 3)
print(f"development terciles of signal-day window RV (n = {len(dev_rv):,}): "
      f"1/3 = {q1:.6f}, 2/3 = {q2:.6f}")

trades without a complete signal-day window: 6 (2017–2023: 6, 2024 onward: 0)
development terciles of signal-day window RV (n = 2,321): 1/3 = 0.005763, 2/3 = 0.010687


In [7]:
with_rv = trades[~missing].copy()
with_rv["activity"] = np.select(
    [with_rv["rv_open"] <= q1, with_rv["rv_open"] <= q2], ["low", "middle"], default="high"
)
by_activity = pd.concat({
    period: grouped(frame, "activity").reindex(["low", "middle", "high"])
    for period, frame in with_rv.groupby("period")
}, axis=1)
by_activity.index.name = "signal-day activity"
show(by_activity)

                    2017–2023                           2024 onward                          
                            n typical_move_% cost_ratio           n typical_move_% cost_ratio
signal-day activity                                                                          
low                       774          2.780      0.108         330          2.240      0.134
middle                    773          4.008      0.075         382          2.133      0.141
high                      774          4.676      0.064         311          2.971      0.101


## Results

**Data.** Daily BTC/USDT 2017-08-17 → 2026-10-20 (3,352 days, no missing days); 3,350 one-day trades with entry days 2017-08-18 → 2026-10-19. Typical move = standard deviation of open-to-next-open log moves (%); cost ratio = 0.30% / typical move.

### Typical one-day move, overall and by year (entry day)

| Year | n | Typical move | Cost ratio |
|---|---|---|---|
| 2017 | 136 | 5.723 | 0.052 |
| 2018 | 365 | 4.398 | 0.068 |
| 2019 | 365 | 3.568 | 0.084 |
| 2020 | 366 | 4.237 | 0.071 |
| 2021 | 365 | 4.238 | 0.071 |
| 2022 | 365 | 3.371 | 0.089 |
| 2023 | 365 | 2.280 | 0.132 |
| 2024 | 366 | 2.757 | 0.109 |
| 2025 | 365 | 2.181 | 0.138 |
| 2026 | 292 | 2.356 | 0.127 |
| Overall | 3,350 | 3.524 | 0.085 |

### By day of week of the entry day

| Entry weekday | 2017–2023 n | Move | Ratio | 2024+ n | Move | Ratio |
|---|---|---|---|---|---|---|
| Monday | 332 | 4.339 | 0.069 | 147 | 2.837 | 0.106 |
| Tuesday | 332 | 3.803 | 0.079 | 146 | 2.529 | 0.119 |
| Wednesday | 332 | 3.943 | 0.076 | 146 | 2.719 | 0.110 |
| Thursday | 332 | 5.092 | 0.059 | 146 | 2.638 | 0.114 |
| Friday | 333 | 3.902 | 0.077 | 146 | 2.704 | 0.111 |
| Saturday | 333 | 2.719 | 0.110 | 146 | 1.232 | 0.244 |
| Sunday | 333 | 3.055 | 0.098 | 146 | 2.034 | 0.147 |

### By signal-day activity around the US open (09:00–10:59 New York)

Signal-day realized volatility of the 8 window returns. Development terciles (n = 2,321): 1/3 = 0.005763, 2/3 = 0.010687. 6 trades (all 2017–2023) have no complete window and are excluded.

| Activity | 2017–2023 n | Move | Ratio | 2024+ n | Move | Ratio |
|---|---|---|---|---|---|---|
| Low | 774 | 2.780 | 0.108 | 330 | 2.240 | 0.134 |
| Middle | 773 | 4.008 | 0.075 | 382 | 2.133 | 0.141 |
| High | 774 | 4.676 | 0.064 | 311 | 2.971 | 0.101 |